In [1]:
import duckdb
from pathlib import Path

STORE_DIR = Path(r"C:\Users\z3553082\AppData\Local\ciccada\ami_store")
con = duckdb.connect()

# Get phase counted sites

In [2]:
load_phase_counts = con.sql(f"""
    SELECT site_id, COUNT(DISTINCT circuit_id) AS n_load_phases
    FROM read_parquet('{(STORE_DIR / "ami_meter").as_posix()}/dt_month=*/*.parquet', hive_partitioning=1)
    GROUP BY site_id
""").df()

pv_allocation = con.sql(f"""
    SELECT site_id, pv_allocation_method, COUNT(*) AS n_rows
    FROM read_parquet('{(STORE_DIR / "ami_raw_phaseseparate").as_posix()}/dt_month=*/*.parquet', hive_partitioning=1)
    GROUP BY site_id, pv_allocation_method
""").df()

# one row per site: its (only, or dominant) allocation method
site_method = (
    pv_allocation.sort_values("n_rows", ascending=False)
    .drop_duplicates("site_id")[["site_id", "pv_allocation_method"]]
)

combined = load_phase_counts.merge(site_method, on="site_id", how="left")

load3_pv3 = combined[(combined.n_load_phases == 3) & (combined.pv_allocation_method == "direct_matched_circuit")]
load1_pv_unmatched = combined[(combined.n_load_phases == 1) & (combined.pv_allocation_method == "equal_split_across_load_phases")]
load3_pv_unmatched = combined[(combined.n_load_phases == 3) & (combined.pv_allocation_method == "equal_split_across_load_phases")]

print(len(load3_pv3), len(load1_pv_unmatched), len(load3_pv_unmatched))
load3_pv3.site_id.tolist()[:10]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

1043 51 689


[1129008036,
 1463277795,
 1607404388,
 205394950,
 652101890,
 239192218,
 127897232,
 2011825833,
 1619142370,
 1501056123]

In [3]:
load1_pv_unmatched

,site_id,n_load_phases,pv_allocation_method
94,590546285,1,equal_split_across_load_phases
304,247383448,1,equal_split_across_load_phases
368,1871185603,1,equal_split_across_load_phases
512,1800618365,1,equal_split_across_load_phases
567,11892611,1,equal_split_across_load_phases
751,1770915585,1,equal_split_across_load_phases
797,617851326,1,equal_split_across_load_phases
966,933964709,1,equal_split_across_load_phases
1269,1059725896,1,equal_split_across_load_phases
1526,1774788193,1,equal_split_across_load_phases


# Datasets check

## AMI METER

In [ ]:
TABLE = "ami_meter"

list_random = con.sql(f"""
    SELECT DISTINCT site_id FROM read_parquet(
        '{(STORE_DIR / TABLE).as_posix()}/dt_month=*/*.parquet', hive_partitioning=1)
    LIMIT 20
""").df()

list_random

In [ ]:
# Example A: (1 pv_site_net + 1 ac_load_net)
SITE_ID = 931508817

# Example B: (3 pv_site_net + 3 ac_load_net)
# SITE_ID = 941652668

# Example C: (3 pv_site_net + 1 ac_load_net)
# SITE_ID = 1252334806

# Example D: (1 pv_site_net + 3 ac_load_net)
# SITE_ID = 771838743

# Site with load_other
SITE_ID = 1030834588

In [ ]:
def read_table(table_name, site_id=SITE_ID):
    return con.sql(f"""
        SELECT * FROM read_parquet('{(STORE_DIR / table_name).as_posix()}/dt_month=*/*.parquet',
                                    hive_partitioning=1)
        WHERE site_id = {site_id}
        ORDER BY t_stamp
    """).df()

In [ ]:
df_meter = read_table(TABLE)

In [ ]:
df_meter

In [ ]:
from datetime import timedelta

df_meter["t_stamp"] = df_meter["t_stamp"] + timedelta(hours=10)

In [ ]:
df_day_meter = df_meter[(df_meter["t_stamp"] >= "2025-01-15") & (df_meter["t_stamp"] < "2025-01-16")]

In [ ]:
df_day_meter

In [ ]:
df_day_meter['circuit_id'].unique()

In [ ]:
import matplotlib.pyplot as plt

fig, (ax_voltage, ax_power) = plt.subplots(
    2, 1,
    sharex=True,
    figsize=(12, 8),
    constrained_layout=True,
    gridspec_kw={"height_ratios": [1, 2]},
)

df_day_meter.plot(
    x="t_stamp",
    y="V",
    kind="line",
    ax=ax_voltage,
    color="purple",
    title="Voltage and Power Measurements",
    ylabel="Voltage (V)",
)

df_day_meter.plot(
    x="t_stamp",
    y=["P_kw", "Q_kvar", "S_kva"],
    kind="line",
    ax=ax_power,
    color=["blue", "orange", "green"],
    ylabel="Power",
)

ax_power.set_xlabel("Timestamp")

In [ ]:
fig, (ax_voltage, ax_power) = plt.subplots(
    2, 1,
    sharex=True,
    figsize=(12, 8),
    constrained_layout=True,
    gridspec_kw={"height_ratios": [1, 2]},
)

# Plot voltage by circuit_id
for circuit_id in df_day_meter['circuit_id'].unique():
    mask = df_day_meter['circuit_id'] == circuit_id
    ax_voltage.plot(
        df_day_meter.loc[mask, 't_stamp'],
        df_day_meter.loc[mask, 'V'],
        label=f'Circuit {circuit_id}',
        marker='o',
        markersize=2,
    )

ax_voltage.set_ylabel("Voltage (V)")
ax_voltage.set_title("Voltage and Power Measurements")
ax_voltage.legend()
ax_voltage.grid(True)

# Plot power by circuit_id
for circuit_id in df_day_meter['circuit_id'].unique():
    mask = df_day_meter['circuit_id'] == circuit_id
    ax_power.plot(
        df_day_meter.loc[mask, 't_stamp'],
        df_day_meter.loc[mask, 'P_kw'],
        label=f'Circuit {circuit_id} (P)',
        marker='o',
        markersize=2,
    )
    ax_power.plot(
        df_day_meter.loc[mask, 't_stamp'],
        df_day_meter.loc[mask, 'Q_kvar'],
        label=f'Circuit {circuit_id} (Q)',
        marker='s',
        markersize=2,
    )

ax_power.set_ylabel("Power")
ax_power.set_xlabel("Timestamp")
ax_power.legend()
ax_power.grid(True)

## RAW

In [ ]:
df_raw = read_table("ami_raw")

In [ ]:
# df_raw.to_csv("df_raw_example.csv", index=False)

In [ ]:
df_raw["t_stamp"] = df_raw["t_stamp"] + timedelta(hours=10)
df_day_raw = df_raw[(df_raw["t_stamp"] >= "2025-01-15") & (df_raw["t_stamp"] < "2025-01-16")]

In [ ]:
df_day_raw

In [ ]:
df_day_raw["P_kw_deNorm"] = df_day_raw["P_pv_kw"] * df_day_raw["S_99"]
df_day_raw["Q_kvar_deNorm"] = df_day_raw["Q_pv_kvar"] * df_day_raw["S_99"]

In [ ]:
fig, (ax_voltage, ax_power) = plt.subplots(
    2, 1,
    sharex=True,
    figsize=(12, 8),
    constrained_layout=True,
    gridspec_kw={"height_ratios": [1, 2]},
)

df_day_raw.plot(
    x="t_stamp",
    y="V_pv",
    kind="line",
    ax=ax_voltage,
    color="purple",
    title="Voltage and Power Measurements",
    ylabel="Voltage (V)",
)

df_day_raw.plot(
    x="t_stamp",
    y=["P_pv_kw", "Q_pv_kvar"],
    kind="line",
    ax=ax_power,
    color=["blue", "orange", "green"],
    ylabel="Power",
)

ax_power.set_xlabel("Timestamp")

In [ ]:
fig, (ax_voltage, ax_power) = plt.subplots(
    2, 1,
    sharex=True,
    figsize=(12, 8),
    constrained_layout=True,
    gridspec_kw={"height_ratios": [1, 2]},
)

df_day_raw.plot(
    x="t_stamp",
    y="V_load",
    kind="line",
    ax=ax_voltage,
    color="purple",
    title="Voltage and Power Measurements",
    ylabel="Voltage (V)",
)

df_day_raw.plot(
    x="t_stamp",
    y=["P_kw", "Q_kvar"],
    kind="line",
    ax=ax_power,
    color=["blue", "orange", "green"],
    ylabel="Power",
)

ax_power.set_xlabel("Timestamp")

In [ ]:
fig, (ax_voltage, ax_power) = plt.subplots(
    2, 1,
    sharex=True,
    figsize=(12, 8),
    constrained_layout=True,
    gridspec_kw={"height_ratios": [1, 2]},
)

df_day_raw.plot(
    x="t_stamp",
    y="V_load",
    kind="line",
    ax=ax_voltage,
    color="purple",
    title="Voltage and Power Measurements",
    ylabel="Voltage (V)",
)

df_day_raw.plot(
    x="t_stamp",
    y=["P_load_kw", "Q_load_kvar"],
    kind="line",
    ax=ax_power,
    color=["blue", "orange", "green"],
    ylabel="Power",
)

ax_power.set_xlabel("Timestamp")

## AMI RAW Phase separate

In [ ]:
df_raw_phase_separate = read_table("ami_raw_phaseseparate")

In [ ]:
# df_raw_phase_separate.to_csv("df_raw_phase_separate_example.csv", index=False)

In [ ]:
df_raw_phase_separate["t_stamp"] = df_raw_phase_separate["t_stamp"] + timedelta(hours=10)
df_day_raw_phase_separate = df_raw_phase_separate[(df_raw_phase_separate["t_stamp"] >= "2025-02-15") & (df_raw_phase_separate["t_stamp"] < "2025-02-16")]

In [ ]:
fig, axes = plt.subplots(
    3, 2,
    figsize=(16, 12),
    constrained_layout=True,
    gridspec_kw={"height_ratios": [1, 1, 1]},
)

# Left column: pv_site_net
pv_data = df_day_raw_phase_separate[df_day_raw_phase_separate['circuit_type'] == 'pv_site_net']
for circuit_id in pv_data['circuit_id'].unique():
    mask = pv_data['circuit_id'] == circuit_id
    axes[0, 0].plot(
        pv_data.loc[mask, 't_stamp'],
        pv_data.loc[mask, 'V'],
        label=f'Circuit {circuit_id}',
        marker='o',
        markersize=2,
    )

axes[0, 0].set_ylabel("Voltage (V)")
axes[0, 0].set_title("PV Site - Voltage")
axes[0, 0].legend()
axes[0, 0].grid(True)

for circuit_id in pv_data['circuit_id'].unique():
    mask = pv_data['circuit_id'] == circuit_id
    axes[1, 0].plot(
        pv_data.loc[mask, 't_stamp'],
        pv_data.loc[mask, 'P_kw_signed'],
        label=f'Circuit {circuit_id} (P)',
        marker='o',
        markersize=2,
    )

axes[1, 0].set_ylabel("Power (kW)")
axes[1, 0].set_title("PV Site - Active Power")
axes[1, 0].legend()
axes[1, 0].grid(True)

for circuit_id in pv_data['circuit_id'].unique():
    mask = pv_data['circuit_id'] == circuit_id
    axes[2, 0].plot(
        pv_data.loc[mask, 't_stamp'],
        pv_data.loc[mask, 'Q_kvar_signed'],
        label=f'Circuit {circuit_id} (Q)',
        marker='s',
        markersize=2,
    )

axes[2, 0].set_ylabel("Reactive Power (kvar)")
axes[2, 0].set_xlabel("Timestamp")
axes[2, 0].set_title("PV Site - Reactive Power")
axes[2, 0].legend()
axes[2, 0].grid(True)

# Right column: ac_load_net
load_data = df_day_raw_phase_separate[df_day_raw_phase_separate['circuit_type'] == 'ac_load_net']
for circuit_id in load_data['circuit_id'].unique():
    mask = load_data['circuit_id'] == circuit_id
    axes[0, 1].plot(
        load_data.loc[mask, 't_stamp'],
        load_data.loc[mask, 'V'],
        label=f'Circuit {circuit_id}',
        marker='o',
        markersize=2,
    )

axes[0, 1].set_ylabel("Voltage (V)")
axes[0, 1].set_title("AC Load - Voltage")
axes[0, 1].legend()
axes[0, 1].grid(True)

for circuit_id in load_data['circuit_id'].unique():
    mask = load_data['circuit_id'] == circuit_id
    axes[1, 1].plot(
        load_data.loc[mask, 't_stamp'],
        load_data.loc[mask, 'P_kw_signed'],
        label=f'Circuit {circuit_id} (P)',
        marker='o',
        markersize=2,
    )

axes[1, 1].set_ylabel("Power (kW)")
axes[1, 1].set_title("AC Load - Active Power")
axes[1, 1].legend()
axes[1, 1].grid(True)

for circuit_id in load_data['circuit_id'].unique():
    mask = load_data['circuit_id'] == circuit_id
    axes[2, 1].plot(
        load_data.loc[mask, 't_stamp'],
        load_data.loc[mask, 'Q_kvar_signed'],
        label=f'Circuit {circuit_id} (Q)',
        marker='s',
        markersize=2,
    )

axes[2, 1].set_ylabel("Reactive Power (kvar)")
axes[2, 1].set_xlabel("Timestamp")
axes[2, 1].set_title("AC Load - Reactive Power")
axes[2, 1].legend()
axes[2, 1].grid(True)

# TS direct query

In [ ]:
from pathlib import Path

import duckdb

base_path = Path(
    r"C:\Users\z3553082\AppData\Local\ciccada\ami_store\ami_extract"
)
parquet_pattern = (
    f"{base_path.as_posix()}/dt_month=*/part-load-*"
)

columns = duckdb.query(f"""
    SELECT *
    FROM read_parquet(
        '{parquet_pattern}',
        hive_partitioning = true
    )
    LIMIT 5
""").df().columns

print(columns)

In [ ]:
site_id = 931508817
circuit_id = 515629
month = "2025-02"

In [ ]:
query = f"""
    SELECT *
    FROM read_parquet(
        '{parquet_pattern}',
        hive_partitioning = true
    )
    WHERE dt_month = '2025-07'
      AND circuit_id = 515629
    ORDER BY t_stamp
"""

july_df = duckdb.query(query).df()

print(july_df.shape)
july_df.head()

In [ ]:
duckdb.query(f"""
    SELECT DISTINCT circuit_id
    FROM read_parquet(
        '{parquet_pattern}',
        hive_partitioning = true
    )
    ORDER BY circuit_id
    LIMIT 50
""").df()

# Query to pre-processed parquet

In [ ]:
import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_current, *_current.parents) if (p / "bms_sa_review").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate the CICCADA repository root from {_current}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from bms_sa_review.synthetic_ami_creation.config import ami_config as Config

print(Config.STORE_DIR)

In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect()

def read_table(table_name, site_id=None, columns="*"):
    where = f"WHERE site_id = {site_id}" if site_id is not None else ""
    return con.sql(f"""
        SELECT {columns} FROM read_parquet(
            '{(Config.STORE_DIR / table_name).as_posix()}/dt_month=*/*.parquet',
            hive_partitioning=1)
        {where}
        ORDER BY t_stamp
    """).df()

In [ ]:
# how many sites/rows landed in each table
for t in ("ami_raw", "ami_meter", "ami_raw_phaseseparate"):
    n = con.sql(f"""
        SELECT count(DISTINCT site_id) AS n_sites, count(*) AS n_rows
        FROM read_parquet('{(Config.STORE_DIR / t).as_posix()}/dt_month=*/*.parquet', hive_partitioning=1)
    """).df()
    print(t, n.to_dict("records")[0])

# site's ground truth
site_id = 845609752
df_raw = read_table("ami_raw", site_id)
df_meter = read_table("ami_meter", site_id)
df_phase = read_table("ami_raw_phaseseparate", site_id)

df_raw.head()

In [ ]:
df_meter